# You have been hired by the Kenya Metrological department as a Data scientist. The ICT manager has tasked you with the job of analyzing weather data for major cities in Kenya. You are required to gather, process, and store the data using various data formats and tools available in Pandas.

## Tasks

- Use the OpenWeatherMap API to fetch current weather data for the following Kenyan cities: Nairobi, Mombasa, Kisumu, Nakuru, and Eldoret.
- Store the collected data in a Pandas DataFrame.
- Save the DataFrame to a CSV file.
-  a simple analysis to find the city with the highest temperature.
- Create a report summarizing the weather conditions for each city.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import requests
import pandas as pd

# API key and endpoint
api_key = 'f4aaacf29976bba0a688d364c5e511c3'
base_url = 'https://api.openweathermap.org/data/2.5/weather'

# Kenyan cities with coordinates (lat, lon)
cities = {
    'Nairobi': (1.2921, 36.8219),
    'Mombasa': (-4.0435, 39.6682),
    'Kisumu': (-0.0917, 34.7680),
    'Nakuru': (-0.3167, 36.0833),
    'Eldoret': (0.3167, 35.2833)
}

# List to store city weather data
weather_list = []

# Fetch weather data for each city
for city, (lat, lon) in cities.items():
    params = {
        'lat': lat,
        'lon': lon,
        'units': 'metric',
        'appid': api_key
    }
    response = requests.get(base_url, params=params)
    if response.status_code == 200:
        data = response.json()
        city_weather = {
            'City': city,
            'Temperature (°C)': data['main']['temp'],
            'Humidity (%)': data['main']['humidity'],
            'Pressure (hPa)': data['main']['pressure'],
            'Weather': data['weather'][0]['description'],
            'Wind Speed (m/s)': data['wind']['speed'],
            'Rain (mm last 1h)': data['rain']['1h'] if 'rain' in data and '1h' in data['rain'] else 0
        }
        weather_list.append(city_weather)
    else:
        print(f'Failed to get data for {city}')

# Create DataFrame
weather_df = pd.DataFrame(weather_list)

# Save DataFrame to CSV
weather_df.to_csv('kenya_weather.csv', index=False)

# Report: Summary of all cities
print("Weather Summary Report for Major Kenyan Cities:")
print(weather_df)

In [ ]:
max_temp_row = weather_df.loc[weather_df['Temperature (°C)'].idxmax()]
print(f"City with highest temperature: {max_temp_row['City']} ({max_temp_row['Temperature (°C)']}°C)\n")
print("Weather Summary: " + str(weather_df.shape[0]) + "cities")
weather_df.sort_values(by='Rain (mm last 1h)', ascending=False)


In [ ]:
#analyzing and giving conclusion summarry describing the rains, temperature and humidity
def analyze_weather_data(df):
    summary = "Weather Analysis Summary:\n"
    
    # Overall weather conditions
    avg_temp = df['Temperature (°C)'].mean()
    avg_humidity = df['Humidity (%)'].mean()
    summary += f"- Average Temperature: {avg_temp:.2f}°C\n"
    summary += f"- Average Humidity: {avg_humidity:.2f}%\n"
    
    # Flood alert analysis
    flood_risk_cities = df[df['Rain (mm last 1h)'] >= 20]
    if not flood_risk_cities.empty:
        summary += "- Cities at Flood Risk:\n"
        for _, row in flood_risk_cities.iterrows():
            summary += f"  - {row['City']} with {row['Rain (mm last 1h)']} mm of rain\n"
    else:
        summary += "- No cities currently at high flood risk.\n"
    
    # Actionable insights
    if not flood_risk_cities.empty:
        summary += "Recommended Actions:\n"
        summary += "- Residents in high-risk cities should stay informed through local news and authorities.\n"
        summary += "- Prepare emergency kits and have evacuation plans ready.\n"
        summary += "- Avoid traveling to or through high-risk areas during heavy rainfall.\n"
    
    return summary

analysis_summary = analyze_weather_data(weather_df)
print(analysis_summary)